In [2]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score

In [3]:
DATA_START_SEASON = 2009
MODEL_START_SEASON = 2010
VAL_SEASON = 2024
TEST_SEASON = 2025
SEASONS = list(range(DATA_START_SEASON, TEST_SEASON + 1))


In [4]:
pbp_columns = [
    'game_id',
    'posteam',
    'defteam',
    'epa',
    'success',
    'pass',
    'rush',
    'qb_kneel',
    'qb_spike',
    'two_point_attempt',
    'interception',
    'fumble_lost',
    'sack',
    'qb_dropback',
    'pass_attempt',
    'qb_hit',
    'passer_player_id',
    'passer_player_name',
]

schedule_columns = [
    'game_id',
    'season',
    'week',
    'gameday',
    'game_type',
    'home_team',
    'away_team',
    'home_score',
    'away_score',
    'home_rest',
    'away_rest',
    'location',
    'home_qb_id',
    'away_qb_id',
    'home_qb_name',
    'away_qb_name',
]

In [5]:
pbp = nfl.load_pbp(SEASONS).select(pbp_columns).to_pandas()
schedule = nfl.load_schedules(SEASONS).select(schedule_columns).to_pandas()

In [6]:
TEAM_ABBR_MAP = {
    "SD": "LAC",
    "OAK": "LV",
    "STL": "LA",
}
pbp[["posteam", "defteam"]] = (
    pbp[["posteam", "defteam"]]
    .replace(TEAM_ABBR_MAP)
)
schedule[["home_team", "away_team"]] = (
    schedule[["home_team", "away_team"]]
    .replace(TEAM_ABBR_MAP)
)

In [7]:
scrimmage_plays = pbp.loc[
    pbp['posteam'].notna()
    & pbp['defteam'].notna()
    & pbp['epa'].notna()
    & pbp['success'].notna()
    & (pbp['pass'].eq(1) | pbp['rush'].eq(1))
    & pbp['qb_kneel'].fillna(0).ne(1)
    & pbp['qb_spike'].fillna(0).ne(1)
    & pbp['two_point_attempt'].fillna(0).ne(1)
].copy()

In [8]:
offense = (
    scrimmage_plays
    .groupby(['game_id', 'posteam'])
    .agg(
        off_epa = ('epa', 'mean'),
        off_success_rate = ('success', 'mean')
    )
    .reset_index()
    .rename(columns={'posteam': 'team'})
)

In [9]:
passing = (
    scrimmage_plays.loc[scrimmage_plays["pass"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(pass_epa = ('epa', 'mean'))
    .reset_index()
    .rename(columns={'posteam': 'team'})
)
rush = (
    scrimmage_plays.loc[scrimmage_plays["rush"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(rush_epa = ('epa', 'mean'))
    .reset_index()
    .rename(columns={'posteam': 'team'})
)

In [10]:
defense = (
    scrimmage_plays
    .groupby(['game_id', 'defteam'])
    .agg(
        def_epa_allowed = ('epa', 'mean'),
        def_success_rate_allowed = ('success', 'mean'),
    )
    .reset_index()
    .rename(columns={'defteam': 'team'})
)

In [11]:
home_points = (
    schedule[['game_id', 'home_team', 'home_score', 'away_score']]
    .rename(columns={'home_team': 'team', 'home_score': 'points_scored', 'away_score': 'points_allowed'})
)
away_points = (
    schedule[['game_id', 'away_team', 'home_score', 'away_score']]
    .rename(columns={'away_team': 'team', 'home_score': 'points_allowed', 'away_score': 'points_scored'})
)
points = pd.concat([home_points, away_points], ignore_index=True)
points.sort_values(by=['game_id', 'team'], inplace=True)
points.reset_index(drop=True, inplace=True)


In [12]:
points['win'] = np.select(
    [
        points['points_scored'] > points['points_allowed'],
        points['points_scored'] < points['points_allowed'],
    ],
    [1.0, 0.0],
    default=0.5,
)

In [13]:
scrimmage_plays['turnover'] = (
    scrimmage_plays['interception'].fillna(0)
    + scrimmage_plays['fumble_lost'].fillna(0)
).clip(upper=1)

In [14]:
off_turnovers = (
    scrimmage_plays
    .groupby(['game_id', 'posteam'])
    .agg(turnovers = ('turnover', 'sum'))
).reset_index().rename(columns={'posteam': 'team'})

In [15]:
def_takeaways = (
    scrimmage_plays
    .groupby(['game_id', 'defteam'])
    .agg(takeaways = ('turnover', 'sum')
).reset_index().rename(columns={'defteam': 'team'}))

In [16]:
off_pass_protection = (
    scrimmage_plays
    .groupby(['game_id', 'posteam'])
    .agg(
        sacks_allowed = ('sack', 'sum'),
        dropbacks = ('qb_dropback', 'sum'))
).reset_index().rename(columns={'posteam': 'team'})
off_pass_protection['off_sack_rate'] = (
    off_pass_protection['sacks_allowed'] / off_pass_protection['dropbacks']
)

In [17]:
def_sack_rate = (
    scrimmage_plays
    .groupby(['game_id', 'defteam'])
    .agg(
        sacks = ('sack', 'sum'),
        dropbacks = ('qb_dropback', 'sum'))
).reset_index().rename(columns={'defteam': 'team'})
def_sack_rate['def_sack_rate'] = (
    def_sack_rate['sacks'] / def_sack_rate['dropbacks']
)

In [18]:
completed_schedule = schedule.loc[
    schedule['home_score'].notna()
    & schedule['away_score'].notna()
].copy()
completed_schedule['neutral_site'] = (
    completed_schedule['location']
    .astype(str)
    .str.lower()
    .eq('neutral')
    .astype(int)
)
base_columns = [
    'season',
    'week',
    'gameday',
    'game_id',
    'neutral_site',
]
home_rows = completed_schedule[base_columns].copy()
home_rows['team'] = completed_schedule['home_team']
home_rows['opponent'] = completed_schedule['away_team']
home_rows['rest_days'] = completed_schedule['home_rest']
home_rows['is_home'] = 1
away_rows = completed_schedule[base_columns].copy()
away_rows['team'] = completed_schedule['away_team']
away_rows['opponent'] = completed_schedule['home_team']
away_rows['rest_days'] = completed_schedule['away_rest']
away_rows['is_home'] = 0
schedule_team_games_df = pd.concat([home_rows, away_rows], ignore_index=True)

In [19]:
features_df = [
    offense,
    passing,
    rush,
    defense,
    points,
    off_turnovers,
    def_takeaways,
    off_pass_protection[['game_id', 'team', 'off_sack_rate']],
    def_sack_rate[['game_id', 'team', 'def_sack_rate']],
]
team_games_df = schedule_team_games_df.copy()
for feature_df in features_df:
    assert not feature_df.duplicated(
        ["game_id", "team"]
    ).any()

    team_games_df = team_games_df.merge(
        feature_df,
        on=["game_id", "team"],
        how="left",
        validate="one_to_one",
    )

In [20]:
team_game_columns = [
    'season',
    'week',
    'gameday',
    'game_id',
    'team',
    'opponent',
    'off_epa',
    'off_success_rate',
    'pass_epa',
    'rush_epa',
    'def_epa_allowed',
    'def_success_rate_allowed',
    'points_scored',
    'points_allowed',
    'win',
    'turnovers',
    'takeaways',
    'off_sack_rate',
    'def_sack_rate',
    'rest_days',
    'is_home',
    'neutral_site']
team_games_df = (
    team_games_df[team_game_columns]
    .sort_values(by=['gameday', 'game_id', 'team'])
    .reset_index(drop=True))
team_games_df.head()

,season,week,gameday,game_id,team,opponent,off_epa,off_success_rate,pass_epa,rush_epa,...,points_scored,points_allowed,win,turnovers,takeaways,off_sack_rate,def_sack_rate,rest_days,is_home,neutral_site
0,2009,1,2009-09-10,2009_01_TEN_PIT,PIT,TEN,-0.143463,0.388889,0.034029,-0.546853,...,13,10,1.0,3.0,2.0,0.083333,0.027778,7,1,0
1,2009,1,2009-09-10,2009_01_TEN_PIT,TEN,PIT,-0.034998,0.359375,0.052764,-0.171905,...,10,13,0.0,2.0,3.0,0.027778,0.083333,7,0,0
2,2009,1,2009-09-13,2009_01_CHI_GB,CHI,GB,-0.237903,0.380282,-0.275556,-0.189319,...,15,21,0.0,4.0,0.0,0.054054,0.125000,7,0,0
3,2009,1,2009-09-13,2009_01_CHI_GB,GB,CHI,-0.139636,0.370370,-0.064642,-0.248718,...,21,15,1.0,0.0,4.0,0.125000,0.054054,7,1,0
4,2009,1,2009-09-13,2009_01_DAL_TB,DAL,TB,0.342538,0.423077,0.656938,-0.053878,...,34,21,1.0,0.0,0.0,0.034483,0.000000,7,0,0


In [21]:
recent_metrics = [
    'off_epa',
    'off_success_rate',
    'pass_epa',
    'rush_epa',
    'def_epa_allowed',
    'def_success_rate_allowed',
]

In [22]:
for column in recent_metrics:
    team_games_df[f'{column}_last3'] = (
        team_games_df
        .groupby(['season', 'team'])[column]
        .transform(
            lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
            )

In [23]:
for column in recent_metrics:
    team_games_df[f'{column}_last5'] = (
        team_games_df
        .groupby(['season', 'team'])[column]
        .transform(
            lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
            )

In [28]:
EWM_SPAN = 5
for column in recent_metrics:
    team_games_df[f'{column}_ewm'] = (
        team_games_df
        .groupby(['season', 'team'])[column]
        .transform(
            lambda x: x.shift(1).ewm(
                span=EWM_SPAN, adjust=False, min_periods=1).mean()
            )
        )
team_games_df.head()

,season,week,gameday,game_id,team,opponent,off_epa,off_success_rate,pass_epa,rush_epa,...,pass_epa_last5,rush_epa_last5,def_epa_allowed_last5,def_success_rate_allowed_last5,off_epa_ewm,off_success_rate_ewm,pass_epa_ewm,rush_epa_ewm,def_epa_allowed_ewm,def_success_rate_allowed_ewm
0,2009,1,2009-09-10,2009_01_TEN_PIT,PIT,TEN,-0.143463,0.388889,0.034029,-0.546853,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009,1,2009-09-10,2009_01_TEN_PIT,TEN,PIT,-0.034998,0.359375,0.052764,-0.171905,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009,1,2009-09-13,2009_01_CHI_GB,CHI,GB,-0.237903,0.380282,-0.275556,-0.189319,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009,1,2009-09-13,2009_01_CHI_GB,GB,CHI,-0.139636,0.370370,-0.064642,-0.248718,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009,1,2009-09-13,2009_01_DAL_TB,DAL,TB,0.342538,0.423077,0.656938,-0.053878,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
prior_metrics = [
    'off_epa',
    'off_success_rate',
    'pass_epa',
    'rush_epa',
    'def_epa_allowed',
    'def_success_rate_allowed',
]